In [1]:
# imports
import json

import logging

# Set up logging configuration at the top of your notebook or script
logging.basicConfig(
    level=logging.DEBUG,  # Change to DEBUG for more verbosity
    format='%(asctime)s - %(levelname)s - %(message)s'
)

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Coder-32B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
   model_name,
   dtype="auto",
   device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

2025-10-12 08:34:56,279 - DEBUG - Starting new HTTPS connection (1): huggingface.co:443
2025-10-12 08:34:56,396 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/config.json HTTP/1.1" 307 0
2025-10-12 08:34:56,401 - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/config.json HTTP/1.1" 200 0
2025-10-12 08:34:57,256 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

2025-10-12 08:35:06,343 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/generation_config.json HTTP/1.1" 307 0
2025-10-12 08:35:06,349 - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/generation_config.json HTTP/1.1" 200 0
2025-10-12 08:35:06,448 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0
2025-10-12 08:35:06,553 - DEBUG - https://huggingface.co:443 "HEAD /Qwen/Qwen2.5-Coder-32B-Instruct/resolve/main/tokenizer_config.json HTTP/1.1" 307 0
2025-10-12 08:35:06,558 - DEBUG - https://huggingface.co:443 "HEAD /api/resolve-cache/models/Qwen/Qwen2.5-Coder-32B-Instruct/381fc969f78efac66bc87ff7ddeadb7e73c218a7/tokenizer_config.json HTTP/1.1" 200 0
2025-10-12 08:35:06,664 - DEBUG - https://huggingface.co:443 "GET /api/models/Qwen/Qwen2.5-Coder-32B-Instruct/tree/main/additional_

In [3]:
import time

def generate_for_prompt(model, tokenizer, user_prompt, **kwargs):
    def build_messages(user_prompt):
        return [
            {
                "role": "system",
                "content": (
                    "You are an expert assistant specialized in generating accurate and high-quality Java code from natural language descriptions. "
                    "When responding, output only the Java code, with no explanations or comments. "
                    "Do not change or reformat code in the prompt; just continue and return the full code (prompt plus your completion) as one complete Java code block."
                )
            },
            {"role": "user", "content": user_prompt}
        ]

    max_new_tokens = kwargs.get("max_new_tokens", 1024)
    top_p = kwargs.get("top_p", 0.95)
    temperature = kwargs.get("temperature", 0.1)
    top_k = kwargs.get("top_k", 0)
    num_return_sequences = kwargs.get("num_return_sequences", 1)
    do_sample = kwargs.get("do_sample", False)

    messages = build_messages(user_prompt)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Measure execution time
    start_time = time.time()
    
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        top_p=top_p,
        temperature=temperature,
        top_k=top_k
    )

    execution_time = time.time() - start_time
    
    logging.info(f"⏱️ Took {execution_time:.2f} secs")

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(
        generated_ids, skip_special_tokens=True)[0]
    return response

In [4]:
from tqdm import tqdm
import math
from concurrent.futures import ThreadPoolExecutor, as_completed


def generate_for_prompts_v2(model, tokenizer, prompts, chunk_size=4, max_workers=4, **kwargs):
    """
    Generate completions for prompts in parallel, divided into chunks.
    Reports overall progress across all prompts using tqdm.
    """
    total = len(prompts)
    completions = [None] * total
    num_chunks = math.ceil(total / chunk_size)
    logging.info(
        f"Total prompts: {total}, Chunk size: {chunk_size}, Chunks: {num_chunks}")

    def process_chunk(chunk_prompts, chunk_indices):
        chunk_results = []
        for idx, prompt in zip(chunk_indices, chunk_prompts):
            response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
            chunk_results.append((idx, response))
        return chunk_results

    # Prepare chunks
    chunks = [
        (prompts[i:i+chunk_size], list(range(i, min(i+chunk_size, total))))
        for i in range(0, total, chunk_size)
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor, tqdm(total=total, desc="Overall Progress") as pbar:
        futures = {executor.submit(process_chunk, chunk_prompts, chunk_indices): (
            chunk_prompts, chunk_indices) for chunk_prompts, chunk_indices in chunks}
        for future in as_completed(futures):
            chunk_results = future.result()
            for idx, response in chunk_results:
                completions[idx] = response
                pbar.update(1)

    logging.info("All completions finished.")
    return completions

In [5]:
def generate_for_prompts(model, tokenizer, prompts, **kwargs):
    completions = []
    start_time = time.time()  # Record start time
    for idx, prompt in enumerate(prompts):
        logging.info(f"Generating completion for problem {idx+1}/{len(prompts)}")
        response = generate_for_prompt(model, tokenizer, prompt, **kwargs)
        completions.append(response if isinstance(
            response, list) else [response])

        logging.info(f'Completion for problem {idx+1}/{len(prompts)}\n')
        logging.debug(f'Completion for problem {idx+1}/{len(prompts)}:\n{response}\n')
        
        # total time has passed
        elapsed_time = time.time() - start_time
        logging.info(f'Elapsed time: {elapsed_time:.2f} secs')
        logging.info(f'Estimated time remaining: {(elapsed_time/(idx+1))*(len(prompts)-idx+1):.2f} secs')
        logging.info(f'{"-"*40}\n')

    return completions

In [6]:
def generate_for_dataset(model, tokenizer, problems, **kwargs):
    user_prompts = [problem['prompt'] for problem in problems]
    
    completions = generate_for_prompts_v2(model, tokenizer, user_prompts, **kwargs) if kwargs.get(
        'parallel', False) else generate_for_prompts(model, tokenizer, user_prompts, **kwargs)
    
    # Save completions to a JSON file
    output_path = 'completions.json'
    with open(output_path, 'w') as f:
        json.dump(completions, f, indent=4)

    return completions

In [7]:
test_prompt = """
import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
""";

In [8]:
print(generate_for_prompt(model, tokenizer, test_prompt))

2025-10-12 08:35:12,225 - INFO - ⏱️ Took 5.32 secs


import java.util.*;
import java.lang.*;

class Solution {
    /**
        Given a positive floating point number, it can be decomposed into
        and integer part (largest integer smaller than given number) and decimals
        (leftover part always smaller than 1).

        Return the decimal part of the number.
        >>> truncateNumber(3.5)
        0.5
     */
    public double truncateNumber(double number) {        
        return number - Math.floor(number);
    }
}


In [ ]:
import os
print(os.getcwd())
ds_json_path = os.path.join(
    '../../../../../..', 'benchmark/datasets/humaneval-x/humanevalx-java-refined.json')

problems = json.load(open(ds_json_path, 'r'))
logging.info(f'Loaded {len(problems)} problems from "{ds_json_path}"')

# completion (2)
generate_for_dataset(model, tokenizer, problems, parallel=False, chunk_size=2, max_workers=4,
                    do_sample=False, max_new_tokens=1024, top_p=0.95, temperature=0.2, top_k=0)

2025-10-12 08:35:12,232 - INFO - Loaded 164 problems from "../../../../../../benchmark/datasets/humaneval-x/humanevalx-java-refined.json"
2025-10-12 08:35:12,233 - INFO - Generating completion for problem 1/164


/workspace/bigcode-evaluation-harness/benchmark/Qwen2.5-Coder-32B-Instruct/java/improve/pass@1/chat


2025-10-12 08:35:21,452 - INFO - ⏱️ Took 9.22 secs
2025-10-12 08:35:21,453 - INFO - Completion for problem 1/164

2025-10-12 08:35:21,454 - DEBUG - Completion for problem 1:
import java.util.*;
import java.lang.*;

class Solution {
    /**
    Check if in given list of numbers, are any two numbers closer to each other than given threshold.
    >>> hasCloseElements(Arrays.asList(1.0, 2.0, 3.0), 0.5)
    false
    >>> hasCloseElements(Arrays.asList(1.0, 2.8, 3.0, 4.0, 5.0, 2.0), 0.3)
    true
     */
    public boolean hasCloseElements(List<Double> numbers, double threshold) {
        Collections.sort(numbers);
        for (int i = 0; i < numbers.size() - 1; i++) {
            if (Math.abs(numbers.get(i) - numbers.get(i + 1)) < threshold) {
                return true;
            }
        }
        return false;
    }
}

2025-10-12 08:35:21,454 - INFO - Elapsed time: 9.22 secs
2025-10-12 08:35:21,454 - INFO - Estimated time remaining: 1521.54 secs
2025-10-12 08:35:21,455 - INFO - 
----

[['import java.util.*;\nimport java.lang.*;\n\nclass Solution {\n    /**\n    Check if in given list of numbers, are any two numbers closer to each other than given threshold.\n    >>> hasCloseElements(Arrays.asList(1.0, 2.0, 3.0), 0.5)\n    false\n    >>> hasCloseElements(Arrays.asList(1.0, 2.8, 3.0, 4.0, 5.0, 2.0), 0.3)\n    true\n     */\n    public boolean hasCloseElements(List<Double> numbers, double threshold) {\n        Collections.sort(numbers);\n        for (int i = 0; i < numbers.size() - 1; i++) {\n            if (Math.abs(numbers.get(i) - numbers.get(i + 1)) < threshold) {\n                return true;\n            }\n        }\n        return false;\n    }\n}'],
 ['import java.util.*;\nimport java.lang.*;\n\nclass Solution {\n    /**\n    Input to this function is a string containing multiple groups of nested parentheses. Your goal is to\n    separate those group into separate strings and return the list of those.\n    Separate groups are balanced (each open brace is prope